# Laboratori 06: Ekuacionet diferenciale dhe dinamika numerike

**Lënda:** Simulim dhe Metoda Numerike  
**Lidhja me leksionin:** Kapitulli 6  
**Kohëzgjatja e rekomanduar:** 4 orë  
**Botimi:** Versioni i zgjidhur për instruktorin

> **Pyetja udhëheqëse:** Si ndikon zgjedhja e integratorit në saktësinë, stabilitetin dhe ruajtjen e energjisë?

Notebook-u ndahet në dy gjysma të barasvlershme. Gjysma e parë përdor Python; gjysma e dytë përdor MATLAB/Octave. Studenti mund të zgjedhë njërën gjuhë, por duhet të raportojë të njëjtat kontrolle numerike dhe interpretime fizike.


## Rezultatet e të nxënit

Në përfundim, studenti duhet të jetë në gjendje:

- të zbatojë Eulerin dhe Runge--Kutta të rendit të katërt;
- të simulojë rënien me rezistencë;
- të studiojë periodën e lavjerrësit jolinear;
- të krahasojë RK4 me skemën Verlet në problemin e dy trupave;


## Përgatitja teorike


Për problemin $\dot{\mathbf{y}}=\mathbf{f}(t,\mathbf{y})$, Euleri i shtjelluar është $\mathbf{y}_{n+1}=\mathbf{y}_n+h\mathbf{f}(t_n,\mathbf{y}_n)$. RK4 kombinon katër pjerrësi dhe ka gabim global $\mathcal{O}(h^4)$. Për sisteme Hamiltoniane, integratorët simplektikë si Verlet-i kufizojnë më mirë zhvendosjen afatgjatë të energjisë.

Lavjerrësi jolinear plotëson $\ddot\theta+(g/L)\sin\theta=0$. Për kënde jo të vogla, perioda largohet nga $T_0=2\pi\sqrt{L/g}$. Problemi i dy trupave reduktohet në $\ddot{\mathbf r}=-\mu\mathbf r/r^3$.


## Algoritmi dhe kriteret e verifikimit

Përdorim të njëjtën ndërfaqe kohore për rënien dhe lavjerrësin, matim gabimin e energjisë dhe pastaj zbatojmë velocity-Verlet për orbitën Kepleriane.

Në çdo eksperiment raportohen parametrat, madhësia e rrjetës ose e mostrës, kriteri i ndalimit, një mbetje ose ligj ruajtjeje dhe një vlerësim i gabimit.


# Pjesa I — Python

Ekzekutoni qelizat sipas radhës. Ndryshoni vetëm një faktor në çdo eksperiment numerik dhe ruani figurat që mbështesin përfundimin.


### Eksperimenti Python 1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(2026)

from scipy.integrate import solve_ivp


### Eksperimenti Python 2


In [ ]:
def rk4(f,t,y0):
    y=np.empty((len(t),len(np.atleast_1d(y0)))); y[0]=y0
    for n,h in enumerate(np.diff(t)):
        k1=f(t[n],y[n]); k2=f(t[n]+h/2,y[n]+h*k1/2)
        k3=f(t[n]+h/2,y[n]+h*k2/2); k4=f(t[n]+h,y[n]+h*k3)
        y[n+1]=y[n]+h*(k1+2*k2+2*k3+k4)/6
    return y
g=9.81; c=0.25; m=1.0
f_fall=lambda t,y:np.array([y[1],-g-c*y[1]*abs(y[1])/m])
t=np.linspace(0,5,501); y=rk4(f_fall,t,[100,0])
plt.plot(t,y[:,0],label='lartësia'); plt.plot(t,y[:,1],label='shpejtësia'); plt.legend(); plt.show()


### Eksperimenti Python 3


In [ ]:
L=1.0
def pend(t,y): return np.array([y[1],-g/L*np.sin(y[0])])
for theta0 in [0.1,0.8,1.5]:
    y=rk4(pend,np.linspace(0,20,4001),[theta0,0]);
    E=.5*L**2*y[:,1]**2+g*L*(1-np.cos(y[:,0])); print(theta0,(E.max()-E.min())/E.mean())


### Eksperimenti Python 4


In [ ]:
def verlet_orbit(dt=0.002,n=12000):
    r=np.zeros((n,2)); v=np.zeros((n,2)); r[0]=[1,0]; v[0]=[0,.82]
    acc=lambda q:-q/np.linalg.norm(q)**3
    for k in range(n-1):
        vh=v[k]+.5*dt*acc(r[k]); r[k+1]=r[k]+dt*vh; v[k+1]=vh+.5*dt*acc(r[k+1])
    return r,v
r,v=verlet_orbit(); E=.5*np.sum(v*v,axis=1)-1/np.linalg.norm(r,axis=1)
fig,axs=plt.subplots(1,2,figsize=(10,4)); axs[0].plot(r[:,0],r[:,1]); axs[0].set_aspect('equal'); axs[1].plot(E-E[0]); axs[1].set_ylabel('$E-E_0$'); plt.show()


## Interpretimi i rezultateve të Python-it

Përgjigjuni me fjali të plota:

1. Cili rezultat verifikon drejtpërdrejt algoritmin?
2. Në cilin regjim rezultati është i pavarur nga hapi, madhësia e mostrës ose gjendja fillestare?
3. Cili është burimi dominues i gabimit ose i pacaktueshmërisë?
4. A mbështetet përfundimi fizik nga figura dhe nga një madhësi numerike?


# Pjesa II — MATLAB/Octave

Qelizat vijuese janë të tipit `raw` që notebook-u të mbetet i hapshëm me kernelin Python. Kopjojini në MATLAB ose Octave, ose ndryshojini në qeliza kodi kur përdorni një kernel MATLAB/Octave.


### Eksperimenti MATLAB/Octave 1


## Krahasimi ndërmjet gjuhëve

Kontrolloni që Python dhe MATLAB/Octave japin rezultate në përputhje me tolerancën e arsyeshme. Ndryshimet në paraqitjen grafike nuk janë dallime shkencore; dallim shkencor është ndryshimi i vlerës, rendit të konvergjencës, mbetjes, energjisë ose intervalit të besimit.


## Produkti që dorëzohet

Dorëzimi duhet të përmbajë:

- notebook-un e ekzekutuar ose skriptin MATLAB/Octave;
- figurat me boshte, njësi dhe legjendë;
- një tabelë të shkurtër me parametrat dhe rezultatet;
- një kontroll verifikimi;
- një paragraf interpretimi fizik;
- një paragraf mbi kufizimet dhe burimet e gabimit;
- funksionet e shkruara nga studenti, të organizuara në mënyrë që të mund të përfshihen më vonë në paketën e tij/saj.


## Kontroll i shpejtë

- A përdor kodi farë të dokumentuar kur ka rastësi?
- A dallohen gabimi numerik dhe pacaktueshmëria e hyrjeve?
- A janë ruajtur njësitë fizike?
- A është kontrolluar rezultati me një kufi, zgjidhje reference, mbetje ose ligj ruajtjeje?
- A mund të riprodhohet rezultati nga një student tjetër?
